In [ ]:
# Implementation from scratch of DQN, for learning purposes

In [ ]:
import gymnasium as gym
from gymnasium.spaces.utils import flatten, flatdim
import math
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
from IPython.display import clear_output
import torch
import torch.nn as nn
import torch.nn.functional as F
import random

In [ ]:
# Create a torch module that has the state space as input, and outputs a softmax of action probabilities
class QModel(nn.Module):
    def __init__(self, state_dims, action_dims):
        super().__init__()
        
        self.linear = nn.Linear(state_dims, 128)
        self.linear2 = nn.Linear(128, action_dims)
    
    def forward(self, x):
        x = F.relu(self.linear(x))
        x = self.linear2(x)
        return x

In [ ]:
class ReplayBuffer():
    def __init__(self):
        self.buffer = []

    def add(self, obs, action, reward, next_obs, done):
        self.buffer.append((obs, action, reward, next_obs, done))

    def sample(self, size):
        if (len(self.buffer) < size):
            return []
        samples = random.sample(self.buffer, size)
        return samples

In [ ]:
test_model = QModel(16, 4)
out = test_model(torch.ones(16))
out

In [ ]:
def make_frozen_lake(render):
    is_slippery = True
    map_name = "4x4"
    render_mode = None
    if render:
        render_mode="rgb_array"
    env = gym.make('FrozenLake-v1', desc=None, map_name=map_name, is_slippery=is_slippery, render_mode=render_mode)
    return env

def make_blackjack(render):
    render_mode = None
    if render:
        render_mode="rgb_array"
    env = gym.make('Blackjack-v1', natural=False, sab=False, render_mode=render_mode)
    return env

def make_taxi(render):
    render_mode = None
    if render:
        render_mode="rgb_array"
    env = gym.make('Taxi-v3', render_mode=render_mode)
    return env

def make_cliffwalking(render):
    render_mode = None
    if render:
        render_mode="rgb_array"
    env = gym.make('CliffWalking-v0', render_mode=render_mode)
    return env

def make_cartpole(render):
    render_mode = None
    if render:
        render_mode="rgb_array"
    env = gym.make('CartPole-v1', render_mode=render_mode)
    return env

def make_lunarlander(render):
    render_mode = None
    if render:
        render_mode="rgb_array"
    env = gym.make("LunarLander-v2", render_mode=render_mode)
    return env

def make_env(render=False):
    return make_frozen_lake(render)

In [ ]:
# Initialise Q(s,a) arbitrarily, for all s element S, a element A(s)
env = make_env()
print(flatdim(env.observation_space))
print(env.action_space.n)

In [ ]:
test_model = QModel(flatdim(env.observation_space), env.action_space.n)
s, _ = env.reset()
print(s)
#t = F.one_hot(torch.tensor(s), env.observation_space.n).to(dtype=torch.float)
#print(t)
#test_model(t)

In [ ]:
def convert_observation(obs, env):
    """Converts an integer observation for ToyText environments to one hot (float)"""
    return torch.as_tensor(flatten(env.observation_space, obs), dtype=torch.float)

In [ ]:
# Execute running our a greedy policy based on Q

def evaluate(env, Q, render = True):
    Q.eval()
    done = False
    G_t = 0

    S, info = env.reset()
    S = convert_observation(S, env)

    if render:
        clear_output(wait=True)
        plt.imshow(env.render())
        plt.show()

    while not done:
        actions = Q(S)
        actions = actions.detach().numpy()
        A = np.argmax(actions)
        S_p, reward, terminated, trunc, info = env.step(A)
        G_t += reward
        S_p = convert_observation(S_p, env)

        S = S_p

        if render:
            clear_output(wait=True)
            plt.imshow(env.render())
            plt.show()

        if terminated or trunc:
            done = True
    
    # return the total return
    return G_t

In [ ]:
start_epsilon = 1.0
gamma = 0.9 # discount factor
lambd = 0.9
alpha = 0.005
replay_buffer_sample_size = 16

epsilon = start_epsilon

Q = QModel(flatdim(env.observation_space), env.action_space.n)
QTarget = QModel(flatdim(env.observation_space), env.action_space.n)
QTarget.load_state_dict(Q.state_dict())
q_target_update_freq = 100
q_target_update_counter = 0

optimizer = torch.optim.Adam(Q.parameters(), 0.001)

replay_buffer = ReplayBuffer()

losses = []

def choose_action(actions, epsilon):
    x = np.random.random()
    
    if x < epsilon:
        # Choose action randomly
        action = np.random.randint(0, len(actions))
    else:
        # Choose action greedily
        best_actions = np.argwhere(actions == np.max(actions))
        action_idx = np.random.randint(0, len(best_actions))
        action = best_actions[action_idx][0]
    return action

def update(env, Q, replay_buffer):
    global q_target_update_counter
    # Update the q target network periodically    
    if q_target_update_counter == q_target_update_freq:
        q_target_update_counter = 0
        QTarget.load_state_dict(Q.state_dict())
    q_target_update_counter += 1

    memory_buffer = replay_buffer.sample(replay_buffer_sample_size)

    for memory in memory_buffer:
        S, A, R, S_next, done = memory
        S = convert_observation(S, env)
        S_next = convert_observation(S_next, env)

        # Choose A' from S' using policy derived from Q (eg, e-greedy)
        actions_p = Q(S)
        #actions_p_d = actions_p.detach().numpy()

        #A_p = choose_action(actions_p, epsilon)
        with torch.no_grad():
            target_actions_p = QTarget(S_next)
        A_p_max = choose_action(target_actions_p.numpy(), 0)

        optimizer.zero_grad()
        if done:
            target = torch.tensor(R, dtype=torch.float)
        else:
            target = R + gamma * target_actions_p[A_p_max].detach()
        #delta = reward + gamma * Q[S_p][A_p_max] - Q[S][A]
        loss = F.mse_loss(actions_p[A], target)
        losses.append(loss.item())
        loss.backward()
        optimizer.step()

def step(env, Q, S, A, E, epsilon, replay_buffer):
    # Take action A using e-greedy, observe R,S'
    S_next, reward, terminated, trunc, info = env.step(A)
    done = terminated or trunc
    replay_buffer.add(S, A, reward, S_next, done)
    S_p = convert_observation(S_next, env)
    
    # # Choose A' from S' using policy derived from Q (eg, e-greedy)
    actions_p = Q(S_p)
    actions_p = actions_p.detach().numpy()
    
    # For sarsa Q update we choose
    A_p = choose_action(actions_p, epsilon)
    # here
    # but for q_learning we choose the max Q(s',a') over all the actions
    # because this is effectively a different policy to the one we are using, it makes
    # q_learning an off policy algorithm. We can do this simply by setting epsilon to 0
    # before calling our choose_action function
    # Note, we must still choose the next action A_p from out current policy for the next step
    # which is returned below
    # A_p = choose_action(actions_p, epsilon)
    # A_p_max = choose_action(actions_p, 0)
    
    update(env, Q, replay_buffer)
    # optimizer.zero_grad()
    # #delta = reward + gamma * Q[S_p][A_p_max] - Q[S][A]
    # loss = F.mse_loss(Q(S)[A], reward + gamma * Q(S_p)[A_p_max])
    # loss.backward()
    # optimizer.step()

    # TODO: Update elgibility trace (if we do td-lambda)
    # E[S][A] = E[S][A] + 1
    
    # for s in range(len(Q)):
    #     for a in range(len(Q[s])):
    #         Q[s][a] = Q[s][a] + alpha * delta * E[s][a]
    #         E[s][a] = gamma * lambd * E[s][a]
    
    return Q, done, S_next, A_p

def episode(env, Q, epsilon, replay_buffer):
    # E(s,a) = 0 for all s element S, a element A(s)
    E = np.zeros((flatdim(env.observation_space), env.action_space.n))
    
    done = False
    S, info = env.reset()
    S_t = convert_observation(S, env)
    
    # Choose initial action
    actions = Q(S_t)
    actions = actions.detach().numpy()
    A = choose_action(actions, epsilon)
    
    # For each step of the episode
    while not done:
        Q, done, S, A = step(env, Q, S, A, E, epsilon, replay_buffer)
    
    return Q

#Q = np.zeros((env.observation_space.n, env.action_space.n))

num_episodes = 4000
evaluation_run_count = 200
av_returns = []
#Q_history = []
#Q_history.append(Q.copy())
Q_diffs = []

def log_stats():
    #Q_history.append(Q.copy())
    s = 0
    for _ in range(evaluation_run_count):
        s += evaluate(env, Q, render=False)
    av_return = s / evaluation_run_count
    av_returns.append(av_return)
    print("Episode: ", episode_idx, "/", num_episodes, ", Epsilon: ", epsilon, ", Average Return: ", av_return)
    # if len(Q_history) > 1:
    #     Q_diff = Q_history[-1] - Q_history[-2]
    #     Q_diff_max = np.max(np.abs(Q_diff))
    #     Q_diffs.append(Q_diff_max)
    #     print(f"Q Max Diff: {Q_diff_max:.2f}")

for episode_idx in tqdm(range(num_episodes)):
    Q = episode(env, Q, epsilon, replay_buffer)
    
    if episode_idx % (num_episodes / 10) == 0:
        log_stats()
        epsilon = epsilon * 0.8

# log stats when we've finished
log_stats()

print("Done")

In [ ]:
plt.plot(losses)

In [ ]:
plt.plot(av_returns)

In [ ]:
plt.plot(Q_diffs)

In [ ]:
Q

In [ ]:
env2 = make_env(render=True)

In [ ]:
# Testing Q
#Q = QModel(env.observation_space.n, env.action_space.n)
G = evaluate(env2, Q, render=True)
print("Total Return: ", G)

In [ ]:
s = 0
final_evaluation_run_count = 5000
for _ in range(final_evaluation_run_count):
    s += evaluate(env, Q, render=False)
av_return = s / final_evaluation_run_count
print("Average Return: ", av_return)